In [1]:
!pip install tensorflow keras scikit-learn

Data Processing

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Step 1: Load Data
def preprocess_selected_params(csv_path, seq_len=24):
    # Load data
    df = pd.read_csv(csv_path)

    # Simulate CO if not present
    if 'co' not in df.columns:
        np.random.seed(42)
        df['co'] = np.random.normal(loc=0.8, scale=0.3, size=len(df))

    # Drop rows with missing target or required inputs
    df = df.dropna(subset=["pm2.5", "TEMP", "DEWP", "co"])

    # Keep only selected features
    df = df[["co", "TEMP", "DEWP", "pm2.5"]]

    # Normalize
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df)

    joblib.dump(scaler, "scaler.pkl")


    # Create time series sequences
    X, y = [], []
    for i in range(len(scaled_data) - seq_len):
        X.append(scaled_data[i:i+seq_len])
        y.append(scaled_data[i+seq_len, 3])  # pm2.5 is at index 3

    return np.array(X), np.array(y)




Model parameters

In [3]:
# Step 3: Process Data
SEQ_LEN = 24
X, y = preprocess_selected_params("PRSA_data_2010.1.1-2014.12.31.csv", seq_len=SEQ_LEN)

# Step 4: Train/Test Split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Step 5: Define and Train LSTM
model = Sequential()
model.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

c:\Users\Hp\miniconda3\envs\ai-bootcamp\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Train model

In [4]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1,
          callbacks=[EarlyStopping(patience=3)])

Epoch 1/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0013 - val_loss: 5.6976e-04
Epoch 2/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 7.0207e-04 - val_loss: 4.8101e-04
Epoch 3/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 6.3397e-04 - val_loss: 4.7461e-04
Epoch 4/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 6.2566e-04 - val_loss: 4.6031e-04
Epoch 5/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 6.1496e-04 - val_loss: 5.7116e-04
Epoch 6/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 6.1638e-04 - val_loss: 4.5735e-04
Epoch 7/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 6.0548e-04 - val_loss: 4.6070e-04
Epoch 8/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 6.0881e-04 - val_loss: 5.1416e-04
Epoch 9/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 6.0463e-04 - val_loss: 5.5736e-04


Save model

In [5]:
model.save("lstm_model.keras")

Predict

In [6]:
# Load model
model = load_model("lstm_model.keras")

# Predict again
future_prediction = model.predict(X_test)

261/261 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [7]:
future_prediction

array([[0.04660688],
       [0.08009353],
       [0.1710382 ],
       ...,
       [0.00366531],
       [0.00272924],
       [0.00240075]], shape=(8347, 1), dtype=float32)